In [1]:
# 🔹 Install (run once if needed)
# !pip install transformers datasets torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 🔹 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 🔹 2. Dataset (sample for lab)
texts = [
    "I love AI", "I hate bugs",
    "This is amazing", "This is terrible",
    "Fantastic work", "Worst experience",
    "I really enjoyed this", "Not good at all",
    "Excellent performance", "Very bad"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

# 🔹 3. Train-Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# 🔹 4. Tokenization
train_encodings = tokenizer(train_texts, padding=True, truncation=True, return_tensors="pt")
test_encodings = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt")

# 🔹 5. Create Dataset
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    torch.tensor(train_labels)
)

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    torch.tensor(test_labels)
)

# 🔹 6. DataLoader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

# 🔹 7. Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# 🔹 8. Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

# 🔹 9. Training Loop
model.train()
epochs = 3

for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# 🔹 10. Evaluation
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(labels.tolist())

# 🔹 11. Accuracy
accuracy = accuracy_score(true_labels, predictions)
print("\nTest Accuracy:", accuracy)

# 🔹 12. Testing on New Data
test_sentences = ["I love this product", "This is horrible"]
encodings = tokenizer(test_sentences, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(
        input_ids=encodings['input_ids'],
        attention_mask=encodings['attention_mask']
    )

preds = torch.argmax(outputs.logits, dim=1)

print("\nPredictions:")
for text, pred in zip(test_sentences, preds):
    print(f"{text} → {'Positive' if pred.item()==1 else 'Negative'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 3.2113
Epoch 2, Loss: 2.8177
Epoch 3, Loss: 2.7327

Test Accuracy: 0.5

Predictions:
I love this product → Positive
This is horrible → Negative
